# 10 — Analisis hasil V2 yang sudah dijalankan

Notebook ini membaca artefak lengkap dari 06–09, menghitung ulang metrik, lalu membuat laporan/grafik terpisah. Tidak melatih model. Kode/config/protokol training yang telah dibekukan tidak diubah.

Hasil tiga kelas tetap endpoint utama. Analisis biner LVOT–RVOT di sini merupakan pemeriksaan eksploratif setelah hasil tiga kelas tersedia, bukan pengganti hasil utama. Prediksi development OOF pada konfigurasi terpilih memiliki bias seleksi; dipakai untuk diagnosis kesalahan.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd() if (Path.cwd() / "research_v2").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from analysis_v2.analyze_results import analyze
OUT = analyze("20260930_anatomy_fs2000_v2")


In [ ]:
display(pd.read_csv(OUT / "model_summary.csv").round(4))
display(Image(filename=str(OUT / "seed_variation.png")))
display(pd.read_csv(OUT / "seed_patient_stability_summary.csv"))

## Perbandingan class weighting pada kasus yang sama

Bandingkan LR/batch/fold/seed yang sama. Angka ini deskriptif; 45 pasangan per model tidak merupakan 45 replikasi independen karena training set fold beririsan.

In [ ]:
display(pd.read_csv(OUT / "best_by_arm.csv"))
display(pd.read_csv(OUT / "weighting_effect_descriptive.csv"))
display(Image(filename=str(OUT / "class_weighting_matched.png")))

In [ ]:
display(Image(filename=str(OUT / "confusion_oof_and_test.png")))
display(pd.read_csv(OUT / "per_class_diagnostics.csv"))

## Antrean review development bersama ahli ECG

Prioritaskan pasien development yang salah pada ketiga model. Kesalahan model dan confidence tinggi tidak membuktikan label diagnosis salah. Periksa kualitas rekaman, kehadiran PVC pada segmen, relasi sadapan, serta dasar label. Jangan gunakan pasien test untuk memilih desain baru.

In [ ]:
review = pd.read_csv(OUT / "development_patient_review_priority.csv", dtype={"hospital_id":str})
display(review[review.models_wrong == 3])
display(pd.read_csv(OUT / "development_errors_by_sublocation.csv"))

## Audit biner sekunder

Jumlahkan probabilitas dua kelompok RVOT, threshold tetap 0,5, seed 42 primer. Tidak memilih threshold dari test. Hasil ini dihitung setelah melihat hasil utama, sehingga bersifat eksploratif. Tiga kelas yang disepakati tetap dipertahankan.

In [ ]:
secondary = pd.read_csv(OUT / "secondary_binary_audit.csv")
display(secondary[secondary.seed == 42].round(4))

In [ ]:
display(Markdown((OUT / "ANALISIS_V2.md").read_text()))